In [1]:
!pip install langchain langchain-chroma langchain-openai
!pip install beautifulsoup4
!pip install langchain-community
!pip install faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.4/128.4 kB 9.7 MB/s eta 0:0

In [2]:
import getpass
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()
os.environ["OPENAI_API_KEY"] = getpass.getpass()

··········


# Simple LLM Call

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [4]:
answer = llm.invoke("how can langsmith help with testing?")

/usr/local/lib/python3.12/dist-packages/langsmith/client.py:290: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [5]:
print(answer)

content="Langsmith can help with testing by providing tools and frameworks for automated testing, such as unit testing, integration testing, and end-to-end testing. These tools can help streamline the testing process, identify bugs and errors, and ensure the overall quality of the software being tested. Langsmith can also provide testing scripts and test cases to help testers effectively evaluate the functionality and performance of the software. Additionally, Langsmith's language processing capabilities can be used to analyze and understand test results, and provide insights and recommendations for improving test coverage and accuracy." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 15, 'total_tokens': 122, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_na

# That's not very interesting ... and there is no chaining anyways

# Let's Do Some Chaining

## Templating

In [6]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI bot. Your name is {name}."),
    ("human", "Hello, how are you doing?"),
    ("ai", "I'm doing well, thanks!"),
    ("human", "{user_input}"),
])

prompt_value = template.invoke(
    {
        "name": "Bob",
        "user_input": "What is your name?"
    }
)
# Output:
# ChatPromptValue(
#    messages=[
#        SystemMessage(content='You are a helpful AI bot. Your name is Bob.'),
#        HumanMessage(content='Hello, how are you doing?'),
#        AIMessage(content="I'm doing well, thanks!"),
#        HumanMessage(content='What is your name?')
#    ]
#)

In [7]:
for msg in prompt_value.messages:
  print(type(msg).__name__, ":", msg.content)

SystemMessage : You are a helpful AI bot. Your name is Bob.
HumanMessage : Hello, how are you doing?
AIMessage : I'm doing well, thanks!
HumanMessage : What is your name?


In [8]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world class technical documentation writer."),
    ("user", "{input}")
])

In [9]:
for msg in prompt.messages:
  print(type(msg).__name__, ":", msg)

SystemMessagePromptTemplate : prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a world class technical documentation writer.') additional_kwargs={}
HumanMessagePromptTemplate : prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}') additional_kwargs={}


## Chaining

In [10]:
chain = prompt | llm

In [11]:
print(chain.first)

input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a world class technical documentation writer.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [12]:
chain_result = chain.invoke({"input": "how can langsmith help with testing?"})

In [13]:
print(chain_result.content)

Langsmith, a powerful language modeling tool, can be a valuable asset in various testing scenarios. Here are some ways Langsmith can assist with testing:

1. Test Case Generation: Langsmith can help in generating test cases by providing sample inputs, expected outputs, edge cases, and boundary scenarios. This can significantly help in creating comprehensive test suites.

2. Test Data Generation: Langsmith can generate realistic test data that encompasses a wide range of values, helping to ensure thorough testing of the system's functionality.

3. Defect Prediction: By analyzing code and identifying patterns, Langsmith can help in predicting potential defects in the software, allowing testers to focus on critical areas.

4. Test Automation: Langsmith can streamline the process of test automation by generating test scripts, test scenarios, and test data. This can save time and effort in writing and maintaining test automation code.

5. Code Review: Langsmith can assist in code reviews by

In [14]:
print(chain_result.response_metadata)

{'token_usage': {'completion_tokens': 292, 'prompt_tokens': 28, 'total_tokens': 320, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CPwboMKdNDer11aPUwc9muLP6e2vI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}


In [15]:
chain_result = chain.invoke({"input": "how are you?"})

In [16]:
print(chain_result.content)

Hello! I'm here to help you with any technical documentation questions you may have. How can I assist you today?


# More Chaining

In [17]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [18]:
chain = prompt | llm | output_parser

In [19]:
chain_result = chain.invoke({"input": "how can langsmith help with testing?"})

In [20]:
print(chain_result)

Langsmith can be a valuable tool for testing various aspects of software applications. Here are some ways Langsmith can help with testing:

1. **Automated Testing**: Langsmith can generate test cases automatically based on the specifications provided. This can help in quickly generating a large number of test cases to cover different scenarios and edge cases.

2. **Test Oracles**: Langsmith can provide automated test oracles that evaluate the correctness of the system under test based on the expected behavior specified in the requirements. This can help in ensuring that the software behaves as intended.

3. **Regression Testing**: Langsmith can help in automatically creating and running regression tests to ensure that new code changes do not introduce new bugs or regressions in the software.

4. **Load Testing**: Langsmith can assist in generating load tests to simulate heavy user loads on the application and identify performance bottlenecks.

5. **Security Testing**: Langsmith can be 

# Where are the retrieval from **R**AG

In [21]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.smith.langchain.com/user_guide")

docs = loader.load()

In [22]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [23]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(docs)
vector = FAISS.from_documents(documents, embeddings)

In [24]:
from langchain.chains.combine_documents import create_stuff_documents_chain
# create_stuff_documents_chain :
#  Create a chain for passing a list of Documents to a model.



prompt = ChatPromptTemplate.from_template("""Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}""", output_parser = output_parser)

document_chain = create_stuff_documents_chain(llm, prompt)
# document_chain = prompt | llm

In [25]:
from langchain.chains import create_retrieval_chain

retriever = vector.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)
# user_input | retriver | llm
#                      /
#   user_input + prompt

In [26]:
response = retrieval_chain.invoke({"input": "how can langsmith help with testing?"})

In [27]:
print(response["answer"])

# LangSmith offers several features that can help with testing:...

LangSmith can help with testing by providing a platform for observability and evaluation of various aspects of a project, such as tracing projects, datasets, annotation queues, and prompts. It allows for the organization of resources within workspaces, and users can have permissions in a workspace that grant them access to the resources within that workspace. Additionally, LangSmith offers features like data retention, usage limits, and monitoring capabilities that can be beneficial for testing and analyzing the performance of a system or application.
